In [17]:
from halib import *
from halib.filetype import csvfile
workdir = "./zout/perf/__analyze"
os.makedirs(workdir, exist_ok=True)

dirs = fs.list_dirs(workdir)
ls_dirs = [
    os.path.join(workdir, d) for d in dirs if os.path.isdir(os.path.join(workdir, d))
]
if "mt_no_temp" in ls_dirs[0]:
    dir_no_temp = ls_dirs[0]
    dir_temp = ls_dirs[1]
else:
    dir_no_temp = ls_dirs[1]
    dir_temp = ls_dirs[0]

csv1_notemp = os.path.join(dir_no_temp, "[per_video]_raw_metric_src_.csv")
csv2_temp = os.path.join(dir_temp, "[per_video]_raw_metric_src_.csv")

df1 = pd.read_csv(csv1_notemp, sep=';', encoding='utf-8')
df2 = pd.read_csv(csv2_temp, sep=';', encoding='utf-8')
df1.drop_duplicates(inplace=True)
df2.drop_duplicates(inplace=True)
df1.rename(columns={"pred": "pred_no_temp", "correct": "correct_no_temp"}, inplace=True)
df2.rename(columns={"pred": "pred_temp", "correct": "correct_temp"}, inplace=True)

df = pd.merge(df1, df2, on=["video_name", "gt"])
cols = ["video_name", "gt", "pred_no_temp", "pred_temp", "correct_no_temp", "correct_temp"]
df = df[cols]

df["diff"] = (df['correct_no_temp'] - df['correct_temp'])
df["correct_both"] = df.apply(lambda row: row['correct_no_temp'] and row['correct_temp'], axis=1)
df.sort_values(by=["correct_temp"], ascending=False, inplace=True)
df.to_csv(os.path.join(workdir, "compare.csv"), sep=';', encoding='utf-8', index=False)

df1 = df[df["correct_both"] == 0]
df1 = df1.sort_values(by=["diff"], ascending=False)
df1.to_csv(
    os.path.join(workdir, "compare_wrong.csv"),
    sep=";",
    encoding="utf-8",
    index=False,
)

In [18]:
import subprocess

def vstack(video1, video2, output="output.mp4"):
    """
    Stick two videos horizontally using ffmpeg.

    Args:
        video1 (str): Path to the first video.
        video2 (str): Path to the second video.
        output (str): Output file path.
    """
    command = [
        "ffmpeg",
        "-i",
        video1,
        "-i",
        video2,
        "-filter_complex",
        "hstack=inputs=2",
        "-c:v",
        "libx264",
        "-crf",
        "23",
        "-preset",
        "veryfast",
        output,
    ]

    subprocess.run(command, check=True)
    print(f"✅ Output saved to {output}")

assert len(ls_dirs) == 2, "Expected exactly two subdirectories in workdir."

def get_csv_files(dir_path):
    csv_files = fs.filter_files_by_extension(dir_path, ext=".csv")
    csv_files = [f for f in csv_files if f.endswith("_results.csv")]
    return csv_files

csvfiles_notemp = get_csv_files(ls_dirs[0])
csvfiles_temp = get_csv_files(ls_dirs[1])
for f1, f2 in tqdm(list(zip(csvfiles_notemp, csvfiles_temp))):
    # f1name = fs.get_file_name(f1, split_file_ext=True)[0]
    # f1name_target = f1name.replace("_results", "_notemp")
    # f2name = fs.get_file_name(f2, split_file_ext=True)[0]
    # f2name_target = f2name.replace("_results", "_temp")
    # f1_target = os.path.join(workdir, f"{f1name_target}.csv")
    # f2_target = os.path.join(workdir, f"{f2name_target}.csv")
    # fs.copy_file(f1, f1_target)
    # fs.copy_file(f2, f2_target)
    df1 = pd.read_csv(
        f1,
        sep=";",
        encoding="utf-8",
        dtype={"pred_label": str, "elapsed_time": float},
        keep_default_na=False,
    )
    df2 = pd.read_csv(
        f2,
        sep=";",
        encoding="utf-8",
        dtype={"pred_label": str, "elapsed_time": float},
        keep_default_na=False,
    )
    # replace all "skipped" to "None" string in df2
    df2['pred_label'] = df2['pred_label'].replace("skipped", "None")
    df1 = df1[["video", "frame_idx", "class_names", "probs", "pred_label"]]
    df2 = df2[['video', 'frame_idx', 'class_names', 'probs', 'pred_label']]
    df1.rename(columns={"probs": "probs_no_temp", "pred_label": "pred_label_no_temp"}, inplace=True)
    df2.rename(columns={"probs": "probs_temp", "pred_label": "pred_label_temp"}, inplace=True)
    df_merged = pd.merge(df1, df2, on=["video", "frame_idx", "class_names"])
    df_merged['pred_diff'] = df_merged.apply(lambda row: row['pred_label_no_temp'] != row['pred_label_temp'], axis=1)
    df_merged = df_merged[
        [
            "video",
            "frame_idx",
            "pred_diff",
            "pred_label_no_temp",
            "pred_label_temp",
            "class_names",
            "probs_no_temp",
            "probs_temp",
        ]
    ]
    vname = fs.get_file_name(f1, split_file_ext=True)[0].split("_")[0]
    output_csv = os.path.join(workdir, f"{vname}.csv")
    df_merged.to_csv(output_csv, sep=';', encoding='utf-8', index=False)

COPY_VIDEO = True
if COPY_VIDEO:
    def get_out_videos(dir_path):
        videos = fs.filter_files_by_extension(dir_path, ext=".mp4")
        videos = [v for v in videos if v.endswith("_out.mp4")]
        return videos
    videos1 = get_out_videos(ls_dirs[0])
    videos2 = get_out_videos(ls_dirs[1])
    pprint(f"Found {len(videos1)} videos in {ls_dirs[0]}")
    pprint(f"Found {len(videos2)} videos in {ls_dirs[1]}")

    pair_ls_video = list(zip(videos1, videos2))
    for v1, v2 in tqdm(pair_ls_video):
        fname = fs.get_file_name(v1, split_file_ext=True)[0]
        vname = fname.split("_")[0]
        output_path = os.path.join(workdir, f"{vname}.mp4")
        vstack(v1, v2, output=output_path)
pprint("Done")

100%|██████████| 2/2 [00:00<00:00, 54.56it/s]


'Found 2 videos in ./zout/perf/__analyze/MainPC__ds_DFire__mt_no_temp__20250919.154620'

'Found 2 videos in ./zout/perf/__analyze/MainPC__ds_DFire__mt_temp_stabilize__20250919.154902'

  0%|          | 0/2 [00:00<?, ?it/s]ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --en

✅ Output saved to ./zout/perf/__analyze/FP1.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

✅ Output saved to ./zout/perf/__analyze/FP35.mp4


'Done'

In [2]:
from halib.research.perftb import *
from halib.research.perfcalc import PerfCalc

pertb = PerfCalc.gen_perf_report_for_multip_exps(
    indir=r"/mnt/e/NextCloud/paper2_main/zout/perf"
)
pertb.plot("./zout/perf/summary.png")

/mnt/e/NextCloud/paper2_main/.venv/lib/python3.11/site-packages/halib/research/perfcalc.py:215: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, temp_df], ignore_index=True)


'------ Final DataFrame Columns ------'

+----+------------------------------------------+-----------+-------------------+-------------------+--------------------+-----------------------+---------------------------------+--------------+
|    | experiment                               | dataset   |   metric_accuracy |   metric_f1_score |   metric_precision |   metric_recall (TPR) |   metric_FPR (False Alarm Rate) |   metric_FPS |
+====+==========================================+===========+===================+===================+====================+=======================+=================================+==============+
|  0 | MainPC__ds_DFire__mt_no_temp__20250919.1 | DFire     |          0.485714 |          0.590909 |           0.490566 |              0.742857 |                        0.771429 |      33.3937 |
|    | 30008_per_video                          |           |                   |                   |                    |                       |                                 |              |
+----+--------------